# Gaussian DFM unconditional and conditional benchmark

Fit a Gaussian population DFM, then train H on its frozen trajectories. Every conditional distribution assessment compares raw H-guided and event-filtered H-guided samples against conditional truth and DFM rejection. Filtering retains the original event subset without resampling.


## Configuration and reproducibility

Run all cells from the repository root. Standard settings use 100 assets, eight factors, 1,000 DFM epochs and 400 sampling steps. Edit CONFIG_OVERRIDES for DFM settings and the final Gaussian call for H and assessment budgets. Validated independent random streams and validation-selected checkpoints are saved in a fresh timestamped artifact directory. Training-loss and unconditional diagnostics precede H sampling; the conditional assessments always include both guided variants.

After switching branches, reload the notebook from disk and restart its kernel before running all cells. Git checkout does not replace imported Python modules or live configuration objects.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import asdict, replace
import sys, time
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown, Image

ROOT = Path.cwd()
if not (ROOT / "DFM.py").exists():
    raise RuntimeError("Run this notebook with the repository as working directory")
sys.path.insert(0, str(ROOT))
from reference import FrozenDFMReference
from sampling import generate_samples
from synthetic_conditional_truth import make_population


In [ ]:
from synthetic_benchmark.configuration import Config, Streams


In [ ]:
MODE = "standard"  # standard: 1,000 epochs; smoke: quick 15-epoch check
# Optional overrides of the selected preset; uncomment the settings to edit.
CONFIG_OVERRIDES = {
    # "d": 100, "k": 8,
    # "seeds": (42,), "cases": ("gaussian",),
    # "n_train": 12_000, "n_val": 3_000, "n_eval": 3_000,
    # "dfm_epochs": 1_000, "steps": 400,
    # "T": 8.0, "t0": 0.005, "width": 64,
    # "batch": 256, "lr": 0.002, "device": "cpu",
}
CONFIG = replace(Config.preset(MODE), **CONFIG_OVERRIDES)
print(f"DFM training: {CONFIG.dfm_epochs:,} epochs per case and seed")
OUTPUT = ROOT / "artifacts" / "synthetic_unconditional_benchmark" / (MODE + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
display(pd.DataFrame([asdict(Config.preset(mode)) for mode in ("smoke", "standard", "research", "large")]).set_index("mode"))
print("Artifacts:", OUTPUT)

## Gaussian population

Returns follow $X=BF+\varepsilon$, with orthonormal loadings, Gaussian factors with mean $(0.15,0,\ldots,0)$ and diagonal variances $(1.8,0.8,0.4,0.2,0.4/3,0.1,0.08,0.4/6)$ at eight factors. Residual variances range from 0.12 to 0.38. Design seed 912 fixes the loadings and the first loading has cosine 0.85 with the equal-weight portfolio direction. Gaussian coefficients, seeded draws and `diagonal-regime-factors-v2` checkpoint provenance are preserved.


In [ ]:
population = make_population(CONFIG.d, "gaussian", k=CONFIG.k)
print("Regime probabilities:", population.pi)
print("Factor means:\n", population.m)
print("Factor covariances:\n", population.S)
print("Residual variances:", population.D)
display(pd.DataFrame([dict(case=case, **make_population(CONFIG.d, case, k=CONFIG.k).variance_contributions()) for case in ("gaussian",)]))

## DFM training and unconditional sampling

`DFM.py` and `NeuralNets.py` implement
$$s_\theta(r,t)=\alpha_tD_tVg_\zeta(V^TD_tr,t)-D_tr,\qquad
\alpha_t=e^{-t/2},\qquad D_t=\operatorname{diag}(1-e^{-t}+e^{-t}c_i)^{-1}.$$
The loading matrix $V$, sigmoid-parameterized residual variances $c_i\in(0,2)$, and a ReLU MLP with two hidden layers of width `CONFIG.width` are learned jointly. After each Adam update, QR retraction restores orthonormal loading columns. The score network uses the factor decomposition with an MLP; it does not implement the paper's empirical 2D U-Net or enforce its theoretical network bounds.

Training samples $t$ uniformly on $[t_0,T]$ and independent noise $\xi\sim N(0,I_d)$, forms $r_t=\alpha_t r+\sqrt{1-e^{-t}}\,\xi$, and minimizes the batch mean of
$$\left\|s_\theta(r_t,t)+\frac{\xi}{\sqrt{1-e^{-t}}}\right\|^2.$$
Validation uses independent observations with fixed times and noise across epochs. The checkpoint with the smallest validation loss is restored and frozen. Noise near $t_0$ can make the training loss variable, so distribution metrics are also needed.

`FrozenDFMReference` supplies the frozen score to Euler–Maruyama sampling:
$$dY_s=\left[\tfrac12Y_s+s_\theta(Y_s,T-s)\right]ds+dW_s,\qquad 0\le s\le T-t_0,$$
starting from $N(0,I_d)$ and using `CONFIG.steps` steps. Evaluation compares terminal samples with the clean population law. Model fitting, Gaussian initialization at finite $T$, stopping at positive $t_0$, and discretization can all contribute to discrepancies.


In [ ]:
from synthetic_benchmark.configuration import write_json
from synthetic_benchmark.unconditional import (
    fit_dfm, initialize_unconditional_run, prepare_unconditional_case,
    fit_and_save_dfm, evaluate_unconditional_samples,
    save_unconditional_case, save_unconditional_summary,
)


## Unconditional metrics and Monte Carlo reference level

Mean error is $\|\hat m-m\|/(\sqrt d\,s)$ with unconditional RMS asset standard deviation $s=\sqrt{\operatorname{tr}\Sigma/d}$. Covariance and correlation errors are relative Frobenius norms against exact population moments. Portfolio metrics include two-sample and analytic-CDF KS distances, 1-Wasserstein distance, and quantiles. Full-vector sliced W1 averages 64 fixed random unit directions in return coordinates.

MMD uses the RBF kernel $\exp(-\|x-y\|^2/(2\ell^2))$, where $\ell^2$ is the median pairwise squared distance among the first up to 512 truth rows. Its unbiased U-statistic uses the first up to 512 rows of each sample and can be negative. Model and matched-size true-versus-true comparisons share the same truth sample, bandwidth, and projection directions. The true-versus-true comparison indicates finite-sample variability; it is not a formal acceptance test for recovery. KS values are reported descriptively. `marginal_pit_ks_mean` and `marginal_pit_ks_std` summarize per-asset KS statistics against uniformity after applying the exact unconditional marginal CDF. They are computed for both generated and truth-baseline samples and included in seed summaries. These marginal checks do not establish the joint law or test portfolio-event-conditional samples.

The covariance trace ratio measures diversity. Loading error is $\|VV^T-BB^T\|_F/\sqrt{2k}$, which compares subspaces without identifying factor rotations or signs. Event frequencies are evaluated at the population's 10%, 5%, and 1% portfolio quantiles. Saved seed summaries report mean, standard deviation, and count for the main distribution metrics.

The four unconditional panels show the portfolio CDF, portfolio QQ comparison, asset 0 histogram with shared bins, and projections onto the first two true loading directions. These two-dimensional projections are evaluation views of the eight-factor model. The displayed loss curve contains validation loss only; both training and validation losses are saved.


In [ ]:
from synthetic_benchmark.reporting import (
    correlation, compare, plot_comparison, plot_dfm_validation,
    unconditional_recovery_report,
)


## Execute unconditional training and evaluation

For every configured training seed and population case, draw separate training, validation, and two evaluation datasets. Fit and select the DFM, freeze it, and generate `CONFIG.n_eval` unconditional samples. Compare these with one truth draw and use the second truth draw for the matched-size Monte Carlo baseline.

The loop saves checkpoints, samples, metrics, and comparison figures for every case/seed pair. The following tables include all pairs; the displayed figures and validation curves use the first configured seed for each case.


In [ ]:
if tuple(CONFIG.cases) != ("gaussian",):
    raise RuntimeError("Stale simulation configuration: restart the kernel, reload this Gaussian notebook, and run all cells in order.")

initialize_unconditional_run(OUTPUT, CONFIG)
started = time.perf_counter()
rows = []
for seed in CONFIG.seeds:
    for case in ("gaussian",):
        folder, streams, pop = prepare_unconditional_case(OUTPUT, CONFIG, case, seed)
        train = pop.sample(CONFIG.n_train, streams.np("train"))
        val = pop.sample(CONFIG.n_val, streams.np("validation"))
        truth = pop.sample(CONFIG.n_eval, streams.np("truth"))
        truth2 = pop.sample(CONFIG.n_eval, streams.np("truth2"))

        model, history, selected, fit_seconds = fit_and_save_dfm(
            train, val, CONFIG, streams, pop, case, folder,
        )
        reference = FrozenDFMReference(model)
        gen_start = time.perf_counter()
        generated = generate_samples(
            reference, CONFIG.n_eval, num_steps=CONFIG.steps,
            generator=streams.torch("generated", CONFIG.device),
        ).cpu().numpy()
        generation_seconds = time.perf_counter() - gen_start

        result, baseline = evaluate_unconditional_samples(
            generated, truth, truth2, pop, reference, CONFIG, case, seed,
            fit_seconds, generation_seconds,
        )
        rows.extend([result, baseline])
        save_unconditional_case(folder, case, generated, truth, truth2, pop, result, baseline)
metrics, numeric = save_unconditional_summary(rows, OUTPUT, started)


In [ ]:
columns = ["case", "seed", "method", "mean_error", "covariance_error", "correlation_error", "portfolio_ks", "portfolio_w1", "sliced_w1", "mmd2_unbiased"]
display(metrics[columns].round(4))
display(metrics.loc[metrics.method == "DFM", ["case", "seed", "loading_projection_error", "event_0.1", "event_0.05", "event_0.01"]].round(4))

In [ ]:
for case in ("gaussian",):
    folder = OUTPUT / f"{case}_seed{CONFIG.seeds[0]}_primary"
    display(Image(filename=str(folder / "unconditional.png")))
    plot_dfm_validation(folder, case)


## Unconditional recovery from this execution

The generated report uses the first Gaussian training seed and compares its covariance error, portfolio KS, and sliced W1 with the true-versus-true baseline. Its automatic assessment uses whether covariance error exceeds twice the baseline covariance error; this is a descriptive heuristic. Inspect the other metrics, the Gaussian population, and available seed variability before claiming recovery. The report is saved as `conclusions.md` in the run directory.


In [ ]:
primary = metrics[metrics.case == "gaussian"]
b = primary[primary.method == "DFM"].iloc[0]
u0 = primary[primary.method == "truth-vs-truth"].iloc[0]
text = unconditional_recovery_report(b, u0, CONFIG)
display(Markdown(text))
(OUTPUT / "conclusions.md").write_text(text)


## Gaussian H training and conditional assessment

Train H for the portfolio event at the population 5% quantile. Guided paths start from Gaussian proposals accepted with probability `H(0, G(0) x)`. This is the normalized learned-H tilt and is the true conditional initial law only for exact H. `initialization_max_proposals` bounds the total initialization proposals across all output batches; `None` uses `max(100 * n_samples, sample_batch_size)`. Exhaustion raises without a Gaussian fallback, and initialization settings and acceptance counts are saved in `sampling.json`. Training, validation, test, and shuffle seeds must define distinct streams; repeated seeds and CPU Torch seed aliases are rejected before training. Raw H-guided and event-filtered H-guided outputs appear in the sample summary, matched-size W1/KS/sliced-W1 comparisons, four-panel distribution figure and portfolio ECDF. Empty filtered samples remain explicitly labeled as unavailable. Independent truth and DFM rejection comparisons provide Monte Carlo reference levels. Loss curves assess H training before either sample variant exists.


In [ ]:
from synthetic_benchmark.conditional import run_gaussian_conditional_simulation
from synthetic_benchmark.configuration import ConditionalAssessmentConfig


In [ ]:
# Train and assess Gaussian H guidance.
gaussian_results = run_gaussian_conditional_simulation(
    CONFIG, OUTPUT,
    epochs=200,
    n_train=10_000,
    n_validation=2_000,
    n_test=2_000,
    n_samples=CONFIG.n_eval,
    tail=0.05,
    widths=(64, 64),
    initialization_seed=2027,
    lr=1e-3,
    batch_size=64,
    path_batch_size=256,
    train_seeds={"train": 32001, "validation": 32002, "test": 32003, "shuffle": 32004},
    sample_batch_size=256,
    initialization_max_proposals=None,  # Global cap: max(100 * n_samples, sample_batch_size).
    guided_seed=42001,
    truth_seed=42002,
    assessment_settings=ConditionalAssessmentConfig(
        rejection_n=1_000, rejection_batch_size=256,
        rejection_max_proposals=50_000, rejection_reference_max_proposals=50_000,
        rejection_seed=42003, rejection_reference_seed=42004,
        truth_reference_seed=42005, projection_seed=42006, matching_seed=42007,
        projections=64,
    ),
)
gaussian_guided_samples = gaussian_results["guided_samples"]
gaussian_true_conditional_samples = gaussian_results["true_conditional_samples"]
gaussian_filtered_guided_samples = gaussian_results["filtered_guided_samples"]
gaussian_rejection_samples = gaussian_results["rejection_samples"]
gaussian_independent_rejection_samples = gaussian_results["independent_rejection_samples"]
gaussian_independent_true_conditional_samples = gaussian_results["independent_true_conditional_samples"]
gaussian_conditional_metrics = gaussian_results["comparison_metrics"]
